# Transfer Learning — Sign Language MNIST (Laboratorio 1)

**Objetivo:** Aplicar Transfer Learning al dataset de Lenguaje de Señas del Laboratorio 1 (`archive/`), comparando dos estrategias (solo entrenar la última capa vs. fine-tuning de más capas) frente a una CNN personalizada sin pesos preentrenados.

> **Nota:** El notebook original `si3014-transferlearningexcercise.ipynb` usa VGG16 con el dataset de Frutas. Este notebook realiza la misma tarea pero sobre el dataset de signos, adaptado a una arquitectura ResNet-18 (más ligera, ideal para CPU) conservando exactamente los mismos principios de Transfer Learning.

---

## Índice
1. [Análisis del script si3014](#analisis)
2. [Importaciones y configuración](#imports)
3. [Carga y exploración del dataset](#dataset)
4. [Clase Dataset personalizada y Transforms](#dataclass)
5. [Modelo Baseline — CNN sin preentrenamiento](#baseline)
6. [Experimento 1 — Solo última capa (backbone congelado)](#exp1)
7. [Experimento 2 — Fine-tuning (layer4 + cabeza)](#exp2)
8. [Resultados y análisis comparativo](#results)


---
## 1. Análisis del script si3014 <a name="analisis"></a>

El notebook `si3014-transferlearningexcercise.ipynb` implementa Transfer Learning para clasificar 6 tipos de frutas. A continuación se resumen los 4 aspectos clave:

### 1.1 Cómo se carga el modelo preentrenado
```python
from torchvision.models import vgg16, VGG16_Weights
weights = VGG16_Weights.DEFAULT          # pesos ImageNet
vgg_model = vgg16(weights=weights)       # descarga los pesos
```
Se usa **VGG16** entrenado en ImageNet (1000 clases, ~138M parámetros). Los pesos ya capturan características visuales generales (bordes, texturas, formas).

### 1.2 Qué capas se congelan o entrenan
```python
vgg_model.requires_grad_(False)   # congela TODAS las capas
```
En la fase inicial se congelan **todos** los parámetros del backbone. Solo la nueva cabeza de clasificación agrega capas entrenables:
```python
my_model = nn.Sequential(
    vgg_model.features,       # CONGELADO
    vgg_model.avgpool,        # CONGELADO
    nn.Flatten(),
    vgg_model.classifier[0:3],# CONGELADO (primeras 3 capas del classifier)
    nn.Linear(4096, 500),     # ENTRENABLE
    nn.ReLU(),
    nn.Linear(500, N_CLASSES) # ENTRENABLE (6 clases de fruta)
)
```
En la fase de **fine-tuning** (sección 9) se descongela todo el modelo con `vgg_model.requires_grad_(True)` y se usa un learning rate bajo (`1e-4`).

### 1.3 Cómo se prepara el dataset
Las imágenes se leen directamente desde carpetas organizadas por clase (`freshapples/`, `rottenapples/`, …). Se define un `Dataset` personalizado con `glob.glob` para recolectar rutas. Las transformaciones redimensionan a **224×224** (tamaño esperado por VGG16) y convierten a `float32`.

### 1.4 Cómo se realiza el entrenamiento
Se usan las funciones `utils.train()` y `utils.validate()` definidas en `training_utils.py`, que implementan un loop estándar de PyTorch:
- `model.train()` → forward → `loss.backward()` → `optimizer.step()`
- `model.eval()` + `torch.no_grad()` para validación
- **Pérdida:** `CrossEntropyLoss` (adecuada para clasificación multiclase)
- **Optimizador:** `Adam`
- **Épocas:** 15 entrenamiento + 5 fine-tuning


---
## 2. Importaciones y configuración del dispositivo <a name="imports"></a>


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, os

# ─── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ─── Dispositivo ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo : {device}")
print(f"PyTorch     : {torch.__version__}")


---
## 3. Carga y exploración del dataset (Sign Language MNIST) <a name="dataset"></a>

El dataset está en la carpeta `archive/` y corresponde al **Sign Language MNIST**: imágenes 28×28 píxeles en escala de grises que representan 24 letras del alfabeto (se excluyen J y Z por requerir movimiento).

| Campo | Valor |
|-------|-------|
| Datos de entrenamiento | 27 455 muestras |
| Datos de prueba | 7 172 muestras |
| Clases | 24 (letras A–Y, sin J) |
| Formato | CSV con columna `label` + 784 píxeles |
| Tamaño original | 28×28 grayscale |

Las etiquetas originales van de 0 a 24 (omitiendo el 9 = J). Se reasignan a 0–23 para que sean contiguas.


In [ ]:
train_df = pd.read_csv("archive/sign_mnist_train.csv")
test_df  = pd.read_csv("archive/sign_mnist_test.csv")

print(f"Train shape : {train_df.shape}  ({len(train_df):,} muestras, {train_df.shape[1]-1} píxeles)")
print(f"Test  shape : {test_df.shape}")
print(f"\nEtiquetas originales ({len(train_df['label'].unique())} clases):")
print(sorted(train_df["label"].unique()))

# Distribución de clases
class_counts = train_df["label"].value_counts().sort_index()
print(f"\nMuestras por clase (min={class_counts.min()}, max={class_counts.max()}, "
      f"promedio={class_counts.mean():.0f})")


In [ ]:
# Visualizar algunos ejemplos del dataset
LETTER_NAMES = [chr(c) for c in range(ord('A'), ord('Z')+1) if chr(c) not in ('J','Z')]

fig, axes = plt.subplots(4, 6, figsize=(12, 8))
fig.suptitle("Ejemplos del dataset Sign Language MNIST\n(24 letras A–Y, sin J)", fontsize=14)

for i, ax in enumerate(axes.flat):
    sample = train_df[train_df["label"] == sorted(train_df["label"].unique())[i % 24]].iloc[0]
    pixels = sample.drop("label").values.reshape(28, 28).astype(np.uint8)
    label_idx = sorted(train_df["label"].unique())[i % 24]
    letter = LETTER_NAMES[list(sorted(train_df["label"].unique())).index(label_idx)]
    ax.imshow(pixels, cmap="gray")
    ax.set_title(f"'{letter}' (label={label_idx})", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## 4. Clase Dataset, Transforms y DataLoaders <a name="dataclass"></a>

**Diferencias respecto al si3014 (frutas):**
- Las imágenes están en formato CSV (no carpetas de archivos `.png`)
- Son en **escala de grises** → se replica el canal para simular RGB (requerido por ResNet)
- Se redimensionan a **64×64** (en lugar de 224×224) para reducir el cómputo en CPU manteniendo el mismo pipeline de Transfer Learning
- 24 clases en lugar de 6


In [ ]:
# ─── Constantes ───────────────────────────────────────────────────────────────
IMG_SIZE   = 64      # Resize 28×28 → 64×64  (más rápido en CPU; mismo principio que 224×224 en si3014)
BATCH_SIZE = 64
N_CLASSES  = 24

# ─── Reasignación de etiquetas (0-24 sin el 9) → 0-23 ─────────────────────
unique_labels = sorted(train_df["label"].unique())
label_map = {v: i for i, v in enumerate(unique_labels)}  # ej. {0:0, 1:1, ..., 8:8, 10:9, ...}
print("Mapa de etiquetas (primeras 12):", dict(list(label_map.items())[:12]))

# ─── Arrays NumPy ─────────────────────────────────────────────────────────────
def df_to_arrays(df):
    X = df.drop("label", axis=1).values.reshape(-1, 28, 28).astype(np.uint8)
    y = np.array([label_map[v] for v in df["label"].values])
    return X, y

X_train_full, y_train_full = df_to_arrays(train_df)
X_test,       y_test       = df_to_arrays(test_df)

print(f"\nX_train : {X_train_full.shape}   y_train : {y_train_full.shape}")
print(f"X_test  : {X_test.shape}   y_test  : {y_test.shape}")

# ─── Clase Dataset ─────────────────────────────────────────────────────────────
class SignDataset(Dataset):
    """
    Lee imágenes 28×28 grayscale desde arrays NumPy.
    Convierte a pseudo-RGB repitiendo el canal 3 veces (necesario para ResNet/VGG).
    """
    def __init__(self, X, y, transform=None):
        self.X         = X          # (N, 28, 28) uint8
        self.y         = y          # (N,) int
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # (28,28) → (1,28,28) → (3,28,28)  pseudo-RGB
        img = torch.from_numpy(self.X[idx]).unsqueeze(0).repeat(3, 1, 1)
        if self.transform:
            img = self.transform(img)
        return img, int(self.y[idx])

# ─── Transforms ──────────────────────────────────────────────────────────────
# train_tf: con augmentation ligera (como en si3014 sección 7.6)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ConvertImageDtype(torch.float32),
])

# val_tf: sin augmentation
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ConvertImageDtype(torch.float32),
])

print("\nTransforms definidos correctamente.")


In [ ]:
# ─── Split train/val 80-20 (mismo seed en todos los experimentos) ─────────────
N          = len(X_train_full)
train_size = int(0.8 * N)
val_size   = N - train_size
print(f"Train : {train_size} | Val : {val_size} | Test : {len(X_test)}")

def make_split(X, y, train_tf, val_tf):
    """Devuelve subconjuntos train y val con sus transforms correspondientes."""
    full_train = SignDataset(X, y, transform=train_tf)
    full_val   = SignDataset(X, y, transform=val_tf)
    g = torch.Generator().manual_seed(SEED)
    idx = torch.randperm(len(full_train), generator=g)
    tr_idx, va_idx = idx[:train_size], idx[train_size:]
    from torch.utils.data import Subset
    return Subset(full_train, tr_idx), Subset(full_val, va_idx)

def make_loaders(train_ds, val_ds, test_ds):
    nw = min(4, os.cpu_count() or 1)
    kw = dict(num_workers=nw, pin_memory=False, persistent_workers=(nw > 0))
    return (
        DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **kw),
        DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw),
        DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw),
    )

test_ds = SignDataset(X_test, y_test, transform=val_tf)
print("Loaders helper definido.")


---
## 5. Funciones de entrenamiento y evaluación

Siguiendo el mismo patrón que `training_utils.py` del si3014.


In [ ]:
def run_epoch(model, loader, optimizer, criterion, train=True):
    """Un epoch de entrenamiento o evaluación."""
    model.train() if train else model.eval()
    total_loss, correct, seen = 0.0, 0, 0
    ctx = torch.enable_grad if train else torch.no_grad
    with ctx():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad(set_to_none=True)
            output = model(x)
            loss   = criterion(output, y)
            if train:
                loss.backward()
                optimizer.step()
            bs          = y.size(0)
            total_loss += loss.item() * bs
            correct    += (output.argmax(1) == y).sum().item()
            seen       += bs
    return total_loss / seen, correct / seen


def train_model(model, tr_loader, va_loader, optimizer, criterion, epochs, tag=""):
    """Entrena el modelo y devuelve el historial de métricas."""
    history = dict(train_loss=[], train_acc=[], val_loss=[], val_acc=[])
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, tr_loader, optimizer, criterion, train=True)
        va_loss, va_acc = run_epoch(model, va_loader, optimizer, criterion, train=False)
        for k, v in zip(history.keys(), [tr_loss, tr_acc, va_loss, va_acc]):
            history[k].append(round(v, 4))
        print(f"  [{tag}] Epoch {ep:2d}/{epochs}  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
              f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}")
    return history


def evaluate(model, loader, criterion):
    return run_epoch(model, loader, None, criterion, train=False)

print("Funciones de entrenamiento listas.")


---
## 6. Modelo Baseline — CNN personalizada (sin Transfer Learning) <a name="baseline"></a>

Este modelo **no usa ningún peso preentrenado**. Representa lo que se haría en el curso sin Transfer Learning (Laboratorio 1). Sirve de referencia para comparar.

**Arquitectura:**
```
Input (3×64×64)
  → ConvBlock(3→32)   → 32×32×32
  → ConvBlock(32→64)  → 16×16×64
  → ConvBlock(64→128) → 8×8×128
  → Flatten → Linear(8192→512) → ReLU → Dropout(0.5) → Linear(512→24)
```


In [ ]:
class ConvBlock(nn.Module):
    """Bloque Conv+BN+ReLU+MaxPool+Dropout (igual al MyConvBlock de training_utils.py)."""
    def __init__(self, in_ch, out_ch, dropout_p=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout2d(dropout_p),
            nn.MaxPool2d(2, 2),
        )
    def forward(self, x):
        return self.block(x)


class BaselineCNN(nn.Module):
    """CNN entrenada desde cero — sin pesos preentrenados."""
    def __init__(self, n_classes=24):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),    # 64 → 32
            ConvBlock(32,  64),    # 32 → 16
            ConvBlock(64, 128),    # 16 → 8
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, n_classes),
        )
    def forward(self, x):
        return self.head(self.features(x))


baseline_model = BaselineCNN(N_CLASSES).to(device)
total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Baseline CNN — parámetros totales: {total_params:,}")
print(baseline_model)


In [ ]:
EPOCHS_BASELINE = 10

# Loaders con augmentation para train
tr_base, va_base = make_split(X_train_full, y_train_full, train_tf, val_tf)
tr_loader_base, va_loader_base, te_loader_base = make_loaders(tr_base, va_base, test_ds)

criterion_base = nn.CrossEntropyLoss()
opt_base       = Adam(baseline_model.parameters(), lr=1e-3)

print(f"Entrenando Baseline CNN — {EPOCHS_BASELINE} épocas …\n")
hist_base = train_model(baseline_model, tr_loader_base, va_loader_base,
                        opt_base, criterion_base, EPOCHS_BASELINE, tag="Baseline")

base_test_loss, base_test_acc = evaluate(baseline_model, te_loader_base, criterion_base)
print(f"\n  ▶ Baseline — Test Loss: {base_test_loss:.4f}  Test Acc: {base_test_acc:.4f}")


---
## 7. Experimento 1 — Transfer Learning: solo la última capa <a name="exp1"></a>

**Estrategia (igual que si3014 sección 7.3):**
```
ResNet18 (ImageNet) → congela TODO → reemplaza solo el fc final
```

```python
model.requires_grad_(False)          # congela backbone completo
model.fc = nn.Linear(512, 24)        # cabeza nueva — SOLO ESTA se entrena
```

**Parámetros entrenables:** 12,312 / 11,188,824 = **0.11 %** del total

Esto permite aprovechar las características visuales de ImageNet (bordes, formas, texturas) sin riesgo de "olvidar" esos aprendizajes, pero limita la capacidad de adaptación al nuevo dominio.


In [ ]:
# ── Cargar ResNet18 preentrenado ──────────────────────────────────────────────
# Equivalente a: vgg_model = vgg16(weights=VGG16_Weights.DEFAULT) en si3014
model_exp1 = resnet18(weights=ResNet18_Weights.DEFAULT)

# ── Congelar TODAS las capas (igual que si3014 sección 7.3) ───────────────────
model_exp1.requires_grad_(False)
print("Todos los parámetros congelados:", not next(iter(model_exp1.parameters())).requires_grad)

# ── Reemplazar solo la última capa (cabeza de clasificación) ──────────────────
# ResNet18 termina en: fc = Linear(512, 1000)  → reemplazamos por Linear(512, 24)
model_exp1.fc = nn.Linear(512, N_CLASSES)   # esta capa SÍ es entrenable

model_exp1 = model_exp1.to(device)

# ── Verificar parámetros entrenables ──────────────────────────────────────────
trainable  = sum(p.numel() for p in model_exp1.parameters() if p.requires_grad)
total      = sum(p.numel() for p in model_exp1.parameters())
print(f"\nParámetros entrenables : {trainable:,}")
print(f"Parámetros totales     : {total:,}")
print(f"Porcentaje entrenable  : {100*trainable/total:.2f} %")


In [ ]:
EPOCHS_EXP1 = 8

tr_exp1, va_exp1 = make_split(X_train_full, y_train_full, val_tf, val_tf)   # sin aug para TL
tr_loader1, va_loader1, te_loader1 = make_loaders(tr_exp1, va_exp1, test_ds)

criterion1 = nn.CrossEntropyLoss()
opt1       = Adam(model_exp1.parameters(), lr=1e-3)   # Adam solo actualiza params con grad

print(f"Entrenando Experimento 1 (solo última capa) — {EPOCHS_EXP1} épocas …\n")
hist1 = train_model(model_exp1, tr_loader1, va_loader1,
                     opt1, criterion1, EPOCHS_EXP1, tag="Exp1")

exp1_test_loss, exp1_test_acc = evaluate(model_exp1, te_loader1, criterion1)
print(f"\n  ▶ Exp1 — Test Loss: {exp1_test_loss:.4f}  Test Acc: {exp1_test_acc:.4f}")


---
## 8. Experimento 2 — Transfer Learning: Fine-Tuning (layer4 + cabeza) <a name="exp2"></a>

**Estrategia (equivalente al si3014 sección 9 — Unfreeze Model):**

Se sigue un proceso en **dos fases**:

**Fase A — Warm-up** (igual que Exp1): Congelar todo excepto `fc` durante 8 épocas para estabilizar los pesos de la cabeza nueva antes de hacer fine-tuning.

**Fase B — Fine-tuning**: Descongelar `layer4` (el último bloque convolucional de ResNet18, ~8.4M parámetros → 75 % del total) y continuar con learning rate reducido (`1e-4`).

```
model.requires_grad_(False)           # congela todo primero
→ Fase A: entrenar solo fc  (8 épocas, lr=1e-3)
→ Fase B: descongelar layer4 + fc  (6 épocas, lr=1e-4)
```

**Parámetros entrenables en Fase B:** 8,406,040 / 11,188,824 = **75.13 %**


In [ ]:
EPOCHS_WARMUP   = 8
EPOCHS_FINETUNE = 6

# ── Modelo nuevo (misma base que Exp1) ────────────────────────────────────────
model_exp2 = resnet18(weights=ResNet18_Weights.DEFAULT)
model_exp2.requires_grad_(False)
model_exp2.fc = nn.Linear(512, N_CLASSES)
model_exp2 = model_exp2.to(device)

tr_exp2, va_exp2 = make_split(X_train_full, y_train_full, val_tf, val_tf)
tr_loader2, va_loader2, te_loader2 = make_loaders(tr_exp2, va_exp2, test_ds)

criterion2 = nn.CrossEntropyLoss()

# ── Fase A: Warm-up (solo fc) ─────────────────────────────────────────────────
print("=== Fase A: Warm-up (solo fc) ===\n")
opt2a = Adam(model_exp2.fc.parameters(), lr=1e-3)
hist2a = train_model(model_exp2, tr_loader2, va_loader2,
                     opt2a, criterion2, EPOCHS_WARMUP, tag="Exp2-WarmUp")

# ── Fase B: Fine-tuning — descongelar layer4 ─────────────────────────────────
print("\n=== Fase B: Fine-tuning (layer4 + fc) ===\n")
for p in model_exp2.layer4.parameters():
    p.requires_grad = True

trainable2 = sum(p.numel() for p in model_exp2.parameters() if p.requires_grad)
total2     = sum(p.numel() for p in model_exp2.parameters())
print(f"Parámetros entrenables: {trainable2:,} / {total2:,} ({100*trainable2/total2:.2f} %)\n")

opt2b = Adam([
    {"params": model_exp2.layer4.parameters(), "lr": 1e-4},
    {"params": model_exp2.fc.parameters(),     "lr": 1e-4},
])
hist2b = train_model(model_exp2, tr_loader2, va_loader2,
                     opt2b, criterion2, EPOCHS_FINETUNE, tag="Exp2-FineTune")

exp2_test_loss, exp2_test_acc = evaluate(model_exp2, te_loader2, criterion2)
print(f"\n  ▶ Exp2 — Test Loss: {exp2_test_loss:.4f}  Test Acc: {exp2_test_acc:.4f}")


---
## 9. Resultados y análisis comparativo <a name="results"></a>

Los resultados a continuación provienen de la ejecución completa de `run_experiments.py`. Se cargan desde `experiment_results.json` para reproducir las gráficas sin necesidad de reentrenar.


In [ ]:
# ─── Cargar resultados guardados ──────────────────────────────────────────────
with open("experiment_results.json") as f:
    R = json.load(f)

h_base  = R["baseline"]["history"]
h_exp1  = R["exp1"]["history"]
h_exp2a = R["exp2"]["history_warmup"]
h_exp2b = R["exp2"]["history_finetune"]

# Concatenar Exp2: warmup + finetune para las curvas de accuracy
exp2_val_acc = h_exp2a["val_acc"] + h_exp2b["val_acc"]
exp2_tr_acc  = h_exp2a["train_acc"] + h_exp2b["train_acc"]

print("Resultados cargados correctamente.")
print(f"\n  Baseline  — Val Acc final: {h_base['val_acc'][-1]:.4f} | Test Acc: {R['baseline']['test_acc']:.4f}")
print(f"  Exp1      — Val Acc final: {h_exp1['val_acc'][-1]:.4f} | Test Acc: {R['exp1']['test_acc']:.4f}")
print(f"  Exp2 (FT) — Val Acc final: {h_exp2b['val_acc'][-1]:.4f} | Test Acc: {R['exp2']['test_acc']:.4f}")


In [ ]:
# ─── Tabla resumen ────────────────────────────────────────────────────────────
import pandas as pd

summary = pd.DataFrame({
    "Modelo": [
        "Baseline CNN (sin preentrenamiento)",
        "Exp1: TL — solo última capa",
        "Exp2: TL — fine-tuning (layer4 + fc)",
    ],
    "Arquitectura": ["CNN propia", "ResNet18 (frozen)", "ResNet18 (layer4+fc)"],
    "Paráms entrenables": ["~2.2 M", "12,312 (0.11%)", "8,406,040 (75.1%)"],
    "Épocas": [10, 8, "8+6 = 14"],
    "Train Acc final": [
        f"{h_base['train_acc'][-1]:.4f}",
        f"{h_exp1['train_acc'][-1]:.4f}",
        f"{h_exp2b['train_acc'][-1]:.4f}",
    ],
    "Val Acc final": [
        f"{h_base['val_acc'][-1]:.4f}",
        f"{h_exp1['val_acc'][-1]:.4f}",
        f"{h_exp2b['val_acc'][-1]:.4f}",
    ],
    "Test Acc": [
        f"{R['baseline']['test_acc']:.4f}",
        f"{R['exp1']['test_acc']:.4f}",
        f"{R['exp2']['test_acc']:.4f}",
    ],
})

print("=" * 95)
print("RESUMEN COMPARATIVO DE MODELOS — Sign Language MNIST")
print("=" * 95)
print(summary.to_string(index=False))
print("=" * 95)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Curvas de Accuracy — Comparativo de Modelos\n(Sign Language MNIST)", fontsize=14)

# ── Baseline ──────────────────────────────────────────────────────────────────
ax = axes[0]
epochs_b = range(1, len(h_base["train_acc"]) + 1)
ax.plot(epochs_b, h_base["train_acc"], "b-o", label="Train")
ax.plot(epochs_b, h_base["val_acc"],   "r-o", label="Val")
ax.set_title("Baseline CNN (sin preentrenamiento)")
ax.set_xlabel("Época"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax.axhline(R["baseline"]["test_acc"], color="green", linestyle="--", alpha=0.7,
           label=f"Test={R['baseline']['test_acc']:.4f}")
ax.legend()

# ── Exp1 ──────────────────────────────────────────────────────────────────────
ax = axes[1]
epochs_e1 = range(1, len(h_exp1["train_acc"]) + 1)
ax.plot(epochs_e1, h_exp1["train_acc"], "b-o", label="Train")
ax.plot(epochs_e1, h_exp1["val_acc"],   "r-o", label="Val")
ax.set_title("Exp1: TL — Solo última capa (frozen backbone)")
ax.set_xlabel("Época"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax.axhline(R["exp1"]["test_acc"], color="green", linestyle="--", alpha=0.7,
           label=f"Test={R['exp1']['test_acc']:.4f}")
ax.legend()

# ── Exp2 (warmup + finetune concatenados) ─────────────────────────────────────
ax = axes[2]
total_ep2 = len(exp2_tr_acc)
epochs_e2 = range(1, total_ep2 + 1)
ax.plot(epochs_e2, exp2_tr_acc, "b-o", label="Train")
ax.plot(epochs_e2, exp2_val_acc, "r-o", label="Val")
ax.axvline(EPOCHS_WARMUP + 0.5, color="gray", linestyle=":", alpha=0.7, label="Inicio Fine-Tuning")
ax.set_title("Exp2: TL — Fine-Tuning (layer4 + fc)\n[fases: warmup | fine-tune]")
ax.set_xlabel("Época"); ax.set_ylabel("Accuracy")
ax.legend(); ax.set_ylim(0, 1.02); ax.grid(True, alpha=0.3)
ax.axhline(R["exp2"]["test_acc"], color="green", linestyle="--", alpha=0.7,
           label=f"Test={R['exp2']['test_acc']:.4f}")
ax.legend()

plt.tight_layout()
plt.savefig("comparison_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada en comparison_curves.png")


In [ ]:
# ─── Gráfica comparativa: Test Accuracy de los 3 modelos ─────────────────────
models = ["Baseline CNN\n(sin preentrenamiento)", "Exp1: TL\nSolo última capa", "Exp2: TL\nFine-Tuning"]
test_accs = [R["baseline"]["test_acc"], R["exp1"]["test_acc"], R["exp2"]["test_acc"]]
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, test_accs, color=colors, width=0.5, edgecolor="black", linewidth=0.8)
ax.set_ylim(0.6, 1.05)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Comparación de Test Accuracy — Sign Language MNIST", fontsize=13)
ax.axhline(0.92, color="red", linestyle="--", alpha=0.6, label="Umbral 92%")

for bar, val in zip(bars, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f"{val*100:.2f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("test_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 10. Análisis comparativo detallado <a name="results"></a>

### 10.1 Resultados obtenidos

| Modelo | Train Acc | Val Acc | Test Acc | Épocas | Paráms entrenables |
|--------|-----------|---------|----------|--------|-------------------|
| Baseline CNN (sin preentrenamiento) | **79.33 %** | **99.07 %** | **91.01 %** | 10 | ~2.2 M (100%) |
| Exp1: TL — solo última capa | **95.56 %** | **96.47 %** | **79.32 %** | 8 | 12,312 (0.11%) |
| Exp2: TL — fine-tuning (layer4+fc) | **99.64 %** | **99.80 %** | **96.57 %** | 14 (8+6) | 8.4 M (75.1%) |

---

### 10.2 Análisis de Train vs Validation Accuracy

**Baseline CNN:**
- Gap train/val grande: 79.33% vs 99.07%. La CPU-augmentation (flip+rotación) dificultó el overfitting durante el entrenamiento, pero el modelo fue capaz de aprender representaciones útiles.
- El test accuracy cae a 91%, mostrando la distribución diferente entre train y test.

**Experimento 1 (solo última capa):**
- Train y val convergen de forma muy estable (95.6% vs 96.5%) sin overfitting.
- El backbone congelado actúa como un extractor de features fijo. La única capa entrenada es el `fc` final, lo que limita fuertemente la capacidad de adaptar las representaciones al dominio de signos.
- **Test accuracy de solo 79.32%** — El backbone de ResNet, entrenado en ImageNet (fotos naturales), extrae características de bajo nivel útiles, pero no captura la especificidad de los gestos de manos en escala de grises.

**Experimento 2 (fine-tuning layer4 + fc):**
- Logra la convergencia más rápida y más alta: val 99.80% después del fine-tuning.
- Train 99.64% ≈ Val 99.80% → mínimo overfitting.
- **Test accuracy 96.57%**, la mejor de los tres modelos.

---

### 10.3 Transfer Learning vs Modelo sin preentrenamiento

| Aspecto | Baseline CNN | Transfer Learning |
|---------|--------------|-------------------|
| Velocidad de convergencia | Lenta (10 épocas para ~99% val) | Rápida (Exp1 llega a 88% en época 1) |
| Conocimiento inicial | Ninguno (pesos aleatorios) | Features de ImageNet (bordes, texturas) |
| Flexibilidad de adaptación | Total (todos los pesos entrenan) | Controlada (define qué descongelar) |
| Riesgo de overfitting | Moderado | Bajo cuando backbone congelado |
| Test Acc Exp1 vs Baseline | **Baseline gana (91% vs 79%)** | El modelo simple tiene ventaja aquí |
| Test Acc Exp2 vs Baseline | **TL gana (97% vs 91%)** | Fine-tuning supera al modelo propio |

**Conclusión clave:** El Transfer Learning sin fine-tuning (Exp1) fue *peor* que la CNN propia en este dataset. Esto ocurre porque las imágenes de signos (28×28, escala de grises, manos) son muy diferentes de las fotos naturales de ImageNet. Sin adaptar el backbone, las features extraídas no son óptimas. Sin embargo, el **fine-tuning (Exp2) superó claramente al Baseline**, demostrando que ajustar capas profundas es fundamental cuando el dominio de origin y destino difieren.

---

### 10.4 Efecto de entrenar solo la última capa vs. entrenar más capas

**Entrenar solo la última capa (Exp1):**
- ✅ Ventajas: muy rápido, pocos parámetros (~12K), converge sin overfitting, no "olvida" ImageNet
- ❌ Desventajas: las representaciones del backbone son fijas e imperfectas para el dominio → test acc 79%
- Recomendado cuando: el dominio destino es similar al origen (ej: clasificar otro tipo de imágenes naturales)

**Fine-tuning (Exp2):**
- ✅ Ventajas: adapta las representaciones al dominio → test acc 97%
- ❌ Desventajas: más costoso, mayor riesgo de sobrentrenamiento, requiere lr bajo para no destruir el preentrenamiento
- Recomendado cuando: el dominio destino difiere del origen (como este caso: signos ≠ ImageNet)
- Clave: se requiere primero una fase warm-up para estabilizar la cabeza antes de descongelar capas profundas

---

### 10.5 Conclusión general

| | |
|---|---|
| **Mejor test accuracy** | Exp2: Fine-Tuning → **96.57 %** |
| **Más eficiente (params)** | Exp1: Solo última capa → 0.11% params entrenables |
| **Convergencia más rápida** | Exp2 en fase B (fine-tuning): 4 épocas para val >99% |
| **Gap train-test más pequeño** | Exp2: 99.64% train vs 96.57% test |

El Transfer Learning con fine-tuning es la estrategia más efectiva para este problema. La elección de ResNet18 sobre VGG16 fue pragmática (x4 más rápido en CPU) pero conceptualmente equivalente al ejercicio si3014.
